# Task 2: Corruption classifier + hard-routed specialist autoencoders

Runtime → Change runtime type → **T4 GPU**, then run the cells **top to bottom**.

**Disconnects are safe.** Every step saves to Google Drive (`MyDrive/GenAI_A1`): Optuna studies are
restored and continue with the remaining trials, training resumes from `last.pt` (`--resume`), and a
step that already finished returns immediately. After a disconnect, run cells 1 and 2 again and then
re-run the cell that was interrupted.

All runtimes below are **estimates** for a free Colab T4 (total ≈ 2–2.5 h); the real numbers are printed
per epoch and logged to Weights & Biases. Design decisions are explained in `docs/task2_notes.md`.

## 1. Get the code

In [ ]:
# 1. code
REPO_URL = 'https://github.com/fsdev87/generative-ai-A1.git'
import os
if os.path.exists('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone {REPO_URL} /content/repo
%cd /content/repo

## 2. Setup
Mounts Drive, installs packages, prepares the Oxford-IIIT Pet cache, logs in to W&B (Colab secret `WANDB_API_KEY`) and sets the directory variables such as `$OUTPUT_DIR` used below.

In [ ]:
# 2. setup: Drive, packages, data cache, W&B, env vars
from src.common.colab import setup
setup(fs2k=False)   # True for Task 4

## 3. Optuna search for the classifier (estimate: 30–40 min)
Study `task2_classifier`: 30 trials of 12 epochs each. It tunes learning rate, batch size, channel
configuration, convolutions per stage, dropout and weight decay, using validation macro-F1 as the
objective. Weak trials are stopped early by MedianPruner. The best configuration is written to
`$OUTPUT_DIR/task2/task2_classifier_best_config.yaml`. Optional: add `--timeout-min 40` to cap the time.

In [ ]:
!python -m src.task2.optuna_classifier --n-trials 30 --epochs 12

## 4. Final classifier training (estimate: 5–15 min)
Trains with the best configuration and the full schedule (60 epochs, early stopping on validation macro-F1).
It writes `checkpoints/task2/classifier/{last,best}.pt` (Task 3 reuses `best.pt`). W&B group: `task2-classifier-final`.

In [ ]:
!python -m src.task2.train_classifier --config "$OUTPUT_DIR/task2/task2_classifier_best_config.yaml" --resume

## 5. Classifier evaluation (estimate: 2–4 min)
Runs on the test manifest (36,690 entries) and on validation. It reports accuracy, macro and per-class
precision/recall/F1, the normalised confusion matrix, accuracy per type × severity, blur detection vs
blur strength, clean↔blur and clean↔occlusion analyses, and misclassified examples.
Outputs go to `$OUTPUT_DIR/task2/tables` and `$OUTPUT_DIR/task2/figures`.

In [ ]:
!python -m src.task2.evaluate_classifier

## 6. Shared Optuna search for the specialists (estimate: 55–75 min)
Study `task2_specialists`: every trial trains the salt, blur and occlusion specialists for 6 epochs
each, interleaved epoch by epoch. The objective is the mean of their three validation restoration scores.
It tunes learning rate, batch size, channel configuration, bottleneck size and the L1/SSIM weight alpha.
The selected configuration is written to `$OUTPUT_DIR/task2/task2_specialists_best_config.yaml`.
Optional: add `--timeout-min 70` to cap the time.

In [ ]:
!python -m src.task2.optuna_specialists --n-trials 20 --epochs 6

## 7. Final specialists: three independent runs (estimate: 5–15 min each)
Each specialist is trained only on its own corruption, with separate parameters and its own W&B group
(`task2-specialist-{type}`). Checkpoints are written to `checkpoints/task2/specialist_{type}/{last,best}.pt`.

In [ ]:
!python -m src.task2.train_specialist --type salt --config "$OUTPUT_DIR/task2/task2_specialists_best_config.yaml" --resume

In [ ]:
!python -m src.task2.train_specialist --type blur --config "$OUTPUT_DIR/task2/task2_specialists_best_config.yaml" --resume

In [ ]:
!python -m src.task2.train_specialist --type occlusion --config "$OUTPUT_DIR/task2/task2_specialists_best_config.yaml" --resume

## 8. Hard-routing evaluation: oracle vs predicted (estimate: 4–8 min)
Evaluates the full test manifest in both routing modes. It writes `eval/test_records_{oracle,predicted}.csv`,
the standard tables, the oracle-vs-predicted comparison, the misrouting analysis (restoration failures
caused by classifier errors) and its figure.

In [ ]:
!python -m src.task2.evaluate_routing

## 9. ONNX export (estimate: 1–2 min)
Exports `classifier.onnx` (output `probs`) and `specialist_{salt,blur,occlusion}.onnx` to `$ONNX_DIR`.
Each file is checked against PyTorch on 64 real test inputs and fails loudly if they differ. Model cards
(`.json`) include the test metrics from steps 5 and 8.

In [ ]:
!python -m src.task2.export_onnx

## 10. Check the outputs

In [ ]:
!ls "$OUTPUT_DIR/task2/tables" "$OUTPUT_DIR/task2/figures" "$OUTPUT_DIR/task2/eval" "$ONNX_DIR"